# 🚀 Google Colab Training Notebook: HCRResNet28 บน GPU (A-Z & 0-9)
**โครงการ:** ระบบตรวจกระดาษคำตอบและให้คะแนนอัตโนมัติ Rayong MARCO (Template 2)  
**สถาบัน:** มหาวิทยาลัยบูรพา  
---
สมุดงาน (Notebook) นี้ถูกออกแบบมาสำหรับฝึกสอนโมเดล **Lightweight HCRResNet28** บน Google Colab ด้วย GPU ฟรี (T4 GPU):
1. **โมเดลตัวเลข 0–9:** สำหรับสแกนกรอบสีม่วง 42 ช่อง (`models/hcr_resnet28_digits.pt`)
2. **โมเดลตัวอักษร A–Z:** สำหรับตรวจช่องคำตอบ 400 ช่อง (`models/hcr_resnet28_az.pt`)
3. **ทดสอบโมเดล End-to-End ทันทีหลังเทรนเสร็จ** และดาวน์โหลดไฟล์ `.pt` กลับเครื่อง

In [ ]:
# Cell 1: ตรวจสอบการเชื่อมต่อ GPU บน Google Colab
import torch
print(f"PyTorch Version: {torch.__version__}")
if torch.cuda.is_available():
    print(f"✅ กำลังใช้งาน GPU: {torch.cuda.get_device_name(0)}")
    device = torch.device('cuda')
else:
    print("⚠️ ยังไม่ได้เปิด GPU! แนะนำให้ไปที่เมนู Runtime -> Change runtime type -> เลือก T4 GPU")
    device = torch.device('cpu')

In [ ]:
# Cell 2: Clone โค้ดโปรเจกต์และชุดข้อมูลจาก GitHub
!git clone https://github.com/67160328/Computer-Vision.git
%cd Computer-Vision/Lab9
!pip install -q opencv-python numpy pandas matplotlib tqdm
!ls -la models/

In [ ]:
# Cell 3: นิยามสถาปัตยกรรม HCRResNet28 ใน PyTorch
import torch.nn as nn
import torch.nn.functional as F

class ResidualBlock(nn.Module):
    def __init__(self, in_c, out_c, stride=1):
        super().__init__()
        self.conv1 = nn.Conv2d(in_c, out_c, kernel_size=3, stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_c)
        self.conv2 = nn.Conv2d(out_c, out_c, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_c)
        self.shortcut = nn.Sequential()
        if stride != 1 or in_c != out_c:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_c, out_c, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm2d(out_c)
            )

    def forward(self, x):
        out = F.gelu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out += self.shortcut(x)
        return F.gelu(out)

class HCRResNet28(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.in_channels = 32
        self.stem = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, stride=1, padding=1, bias=False),
            nn.BatchNorm2d(32),
            nn.GELU()
        )
        self.stage1 = self._make_layer(64, num_blocks=2, stride=1)
        self.stage2 = self._make_layer(128, num_blocks=2, stride=2)
        self.stage3 = self._make_layer(192, num_blocks=2, stride=2)
        self.pool = nn.AdaptiveAvgPool2d((1, 1))
        self.drop = nn.Dropout(p=0.20)
        self.classifier = nn.Linear(192, num_classes)

    def _make_layer(self, out_channels, num_blocks, stride):
        strides = [stride] + [1] * (num_blocks - 1)
        layers = []
        for s in strides:
            layers.append(ResidualBlock(self.in_channels, out_channels, stride=s))
            self.in_channels = out_channels
        return nn.Sequential(*layers)

    def forward(self, x):
        x = self.stem(x)
        x = self.stage1(x)
        x = self.stage2(x)
        x = self.stage3(x)
        x = self.pool(x)
        x = torch.flatten(x, 1)
        x = self.drop(x)
        return self.classifier(x)

print("✅ นิยามโมเดล HCRResNet28 เรียบร้อยแล้ว!")

## 🎯 1. ฝึกสอนโมเดลตัวเลข 0–9 (10 คลาส)
ฝึกสอนบนชุดข้อมูล `data/Number Handwriter` (21,555 ภาพ) แบ่งข้อมูลแบบ Stratified 80:10:10 พร้อม Data Augmentation เพื่อสแกนกรอบสีม่วง 42 ช่อง

In [ ]:
# Cell 4: ฝึกสอนโมเดลตัวเลข 0-9 (10 คลาส) ด้วย GPU
!python src/recognition/train_digit_hcr.py --epochs 25 --batch-size 128

## 🎯 2. ฝึกสอนโมเดลตัวอักษร A–Z (26 คลาส)
ฝึกสอนบนชุดข้อมูลตัวอักษรลายมือเขียน A–Z พร้อมเทคนิค Exam-Sheet Stroke Augmentation

In [ ]:
# Cell 5: ฝึกสอนโมเดลตัวอักษร A-Z (26 คลาส) ด้วย GPU
!python src/recognition/train_hcr.py --epochs 20 --batch-size 128

## 📊 3. ทดสอบการตรวจข้อสอบทั้งแผ่นทันที (End-to-End Verification)

In [ ]:
# Cell 6: รันการตรวจให้คะแนนกระดาษคำตอบจริง 1 แผ่นเพื่อทดสอบผลลัพธ์
!python test_sheet.py "data/Test/ตัวอย่างที่ 3.jpg"

## 💾 4. ดาวน์โหลดน้ำหนักโมเดล (.pt) กลับสู่คอมพิวเตอร์ของคุณ

In [ ]:
# Cell 7: ดาวน์โหลดไฟล์โมเดลที่เทรนเสร็จแล้ว
from google.colab import files
import os

if os.path.exists('models/hcr_resnet28_digits.pt'):
    print('กำลังดาวน์โหลด models/hcr_resnet28_digits.pt ...')
    files.download('models/hcr_resnet28_digits.pt')

if os.path.exists('models/hcr_resnet28_az.pt'):
    print('กำลังดาวน์โหลด models/hcr_resnet28_az.pt ...')
    files.download('models/hcr_resnet28_az.pt')